## 1. Setup & Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm
import wandb
import matplotlib.pyplot as plt

## 2. Configuration 

In [ ]:
CONFIG = {
    'experiment_name': 'exp_001_deep_cnn_scratch',
    'entity': 'aloktripathi-iit-madras',   
    'project_name': '23f3003225-t12026',     
    'sr': 22050,
    'duration': 29.0,      # Slightly less than 30s to fit uniform frames
    'n_fft': 2048,
    'hop_length': 512,
    'n_mels': 128,
    'batch_size': 16,      # Adjusted for GPU VRAM with deep model
    'epochs': 30,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'seed': 42,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

## 3. Reproducibility

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])

## 4. Paths

In [ ]:
BASE_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TRAIN_DIR = os.path.join(BASE_DIR, "genres_stems")
TEST_DIR = os.path.join(BASE_DIR, "mashups")
TEST_CSV = os.path.join(BASE_DIR, "test.csv")

SUBMISSION_FILE = "submission.csv"

## 5. W&B Init

In [ ]:
!pip install --upgrade wandb

In [ ]:
# W&B INITIALIZATION

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("wandb_api")
    wandb.login(key=wandb_key)

    # wandb.login(key="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")
    
    wandb.init(
        entity=CONFIG['entity'],
        project=CONFIG['project_name'], 
        name=CONFIG['experiment_name'], 
        config=CONFIG
    )
    print(f"✅ W&B Active: {CONFIG['entity']}/{CONFIG['project_name']}")
except Exception as e:
    print(f"❌ W&B Error: {e}")
    print("Running in offline mode.")
    wandb.init(mode="disabled")

print(f"Running on: {CONFIG['device']}")

## 6. Data Preprocessing & Dataset

In [ ]:
# Cell 2: Data Preprocessing & Dataset

class AudioUtil:
    @staticmethod
    def load_and_mix_stems(folder_path, genre, filename_base):
        mix = None
        stems = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
        
        # Construct path: train_dir/genre/filename_base/stem
        # Note: Adjust path logic based on actual unzipped structure
        # Assuming structure: genres_stems/genre/song_folder/stem.wav
        
        song_path = os.path.join(folder_path, genre, filename_base)
        
        for stem in stems:
            stem_path = os.path.join(song_path, stem)
            if os.path.exists(stem_path):
                # Load with librosa
                y, _ = librosa.load(stem_path, sr=CONFIG['sr'], duration=CONFIG['duration'])
                if mix is None:
                    mix = y
                else:
                    # Handle varying lengths if any (truncate to min)
                    min_len = min(len(mix), len(y))
                    mix = mix[:min_len] + y[:min_len]
        
        # Normalize mix
        if mix is not None and np.max(np.abs(mix)) > 0:
            mix = mix / np.max(np.abs(mix))
            
        return mix

    @staticmethod
    def compute_mel_spec(y):
        # Calculate Mel Spectrogram
        melspec = librosa.feature.melspectrogram(
            y=y, 
            sr=CONFIG['sr'], 
            n_fft=CONFIG['n_fft'], 
            hop_length=CONFIG['hop_length'], 
            n_mels=CONFIG['n_mels']
        )
        
        # Convert to Log scale (dB)
        log_melspec = librosa.power_to_db(melspec, ref=np.max)
        
        # Shape: (n_mels, time_steps) -> (128, ~1250)
        return log_melspec

# Prepare Metadata
# We need to list all training folders.
# Assumption: TRAIN_DIR contains folders named by genre, containing subfolders for songs.
all_genres = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
print(f"Genres found: {all_genres}")

train_data = []
# Limit for quick debugging if needed, else run full
for genre in all_genres:
    genre_dir = os.path.join(TRAIN_DIR, genre)
    song_folders = os.listdir(genre_dir)
    for song in song_folders:
        train_data.append({
            'genre': genre,
            'song_name': song,
            'path': os.path.join(genre_dir, song)
        })

df_train = pd.DataFrame(train_data)
encoder = LabelEncoder()
df_train['label'] = encoder.fit_transform(df_train['genre'])

# Split validation (Stratified)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df_train, test_size=0.2, stratify=df_train['label'], random_state=CONFIG['seed'])

print(f"Training Samples: {len(train_df)} | Validation Samples: {len(val_df)}")

class MashupDataset(Dataset):
    def __init__(self, df, base_dir, is_train=True, transform=None):
        self.df = df
        self.base_dir = base_dir
        self.is_train = is_train
        self.transform = transform # Placeholder for augmentation
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Load Audio
        # Training data is structured as stems, Test data is single wav
        if self.is_train:
            # Load and Mix Stems
            y = AudioUtil.load_and_mix_stems(TRAIN_DIR, row['genre'], row['song_name'])
        else:
            # Load Test File (Single Wav)
            # Row has 'id' which maps to filename in mashups folder
            file_path = os.path.join(TEST_DIR, f"{row['id']}")
            y, _ = librosa.load(file_path, sr=CONFIG['sr'], duration=CONFIG['duration'])

        # Pad or Crop to fixed length (ensure consistency for batching)
        target_len = int(CONFIG['sr'] * CONFIG['duration'])
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]

        # 2. Feature Extraction (Log Mel)
        spec = AudioUtil.compute_mel_spec(y)
        
        # 3. To Tensor
        # Input to CNN expects (Channels, Freq, Time)
        spec_tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)
        
        if self.is_train:
            label = torch.tensor(row['label'], dtype=torch.long)
            return spec_tensor, label
        else:
            return spec_tensor, row['id']

# Data Loaders
train_ds = MashupDataset(train_df, TRAIN_DIR, is_train=True)
val_ds = MashupDataset(val_df, TRAIN_DIR, is_train=True)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

## 7. Model Architecture (Deep CNN Scratch)

In [ ]:
class DeepAudioCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(DeepAudioCNN, self).__init__()
        
        # Conv Block Definition
        def conv_block(in_channels, out_channels, pool=True):
            layers = [
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            ]
            if pool:
                layers.append(nn.MaxPool2d(kernel_size=2))
            return nn.Sequential(*layers)
        
        # Architecture: 6 Blocks increasing capacity then compressing
        # Input: (B, 1, 128, 1250)
        
        self.block1 = conv_block(1, 32)      # -> (32, 64, 625)
        self.block2 = conv_block(32, 64)     # -> (64, 32, 312)
        self.block3 = conv_block(64, 128)    # -> (128, 16, 156)
        self.block4 = conv_block(128, 256)   # -> (256, 8, 78)
        self.block5 = conv_block(256, 512)   # -> (512, 4, 39)
        self.block6 = conv_block(512, 512)   # -> (512, 2, 19)
        
        # Global Pooling to handle variable time and reduce parameters
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1)) 
        
        # Classification Head
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
        # Weight Initialization
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        
        x = self.global_pool(x)
        x = self.fc(x)
        return x

# Instantiate
model = DeepAudioCNN(num_classes=len(all_genres)).to(CONFIG['device'])
print(model)

# Verify Output Shape
dummy_input = torch.randn(2, 1, 128, 1250).to(CONFIG['device'])
print(f"Output Shape: {model(dummy_input).shape}")

## 8. Training Loop

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3) 
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_f1

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    val_loss = running_loss / len(loader)
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    return val_loss, val_f1, all_labels, all_preds

# Training Execution
best_f1 = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

print(f"Starting training for {CONFIG['epochs']} epochs...")

for epoch in range(CONFIG['epochs']):
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, CONFIG['device'])
    val_loss, val_f1, _, _ = validate(model, val_loader, criterion, CONFIG['device'])
    
    # Scheduler step
    scheduler.step(val_f1)
    
    # Get current LR manually since verbose is gone
    current_lr = optimizer.param_groups[0]['lr']

    # Logging
    wandb.log({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_f1': train_f1,
        'val_loss': val_loss,
        'val_f1': val_f1,
        'lr': current_lr
    })
    
    history['train_loss'].append(train_loss)
    history['val_f1'].append(val_f1)
    
    print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | Val Loss: {val_loss:.4f} F1: {val_f1:.4f} | LR: {current_lr:.2e}")
    
    # Save Best Model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"{CONFIG['experiment_name']}_best.pth")
        print(f"--> Best Model Saved (F1: {best_f1:.4f})")

print(f"Training Complete. Best Validation F1: {best_f1:.4f}")

## 9. Inference & Submission

In [ ]:
# 1. Load Best Model Weights
checkpoint_path = f"{CONFIG['experiment_name']}_best.pth"
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    print(f"✅ Loaded weights from {checkpoint_path}")
else:
    print("⚠️ Checkpoint not found on disk, using current model weights in memory.")

model.eval()

# 2. Load Test Data Metadata
test_df = pd.read_csv(TEST_CSV)
print(f"Found {len(test_df)} test samples. Generating predictions...")

test_ids = []
test_preds = []

# Target length matching training (e.g., 22050 * 29.0)
TARGET_LEN = int(CONFIG['sr'] * CONFIG['duration'])

# 3. Robust Inference Loop (Row by Row)
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    file_id = str(row['id'])
    
    # Safely handle file extensions
    file_path = os.path.join(TEST_DIR, file_id)
    if not os.path.exists(file_path):
        file_path = f"{file_path}.wav"
        
    try:
        # Load Audio
        y, _ = librosa.load(file_path, sr=CONFIG['sr'], duration=CONFIG['duration'])
        
        # Pad or Crop to exact length
        if len(y) < TARGET_LEN:
            y = np.pad(y, (0, TARGET_LEN - len(y)))
        else:
            y = y[:TARGET_LEN]
            
        # Extract Log-Mel Spectrogram (Using AudioUtil from Cell 2)
        spec = AudioUtil.compute_mel_spec(y)
        
        # Convert to Tensor -> Shape: (1, 1, 128, Time)
        spec_tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(CONFIG['device'])
        
        # Predict
        with torch.no_grad():
            outputs = model(spec_tensor)
            pred_idx = torch.argmax(outputs, dim=1).item()
            
            # Decode label
            pred_genre = encoder.inverse_transform([pred_idx])[0]
            
        test_ids.append(file_id)
        test_preds.append(pred_genre)
        
    except Exception as e:
        # 🔥 CRITICAL FALLBACK: If a file is corrupt, predict 'pop' so the submission doesn't fail
        print(f"⚠️ Error processing {file_id}: {e} -> Defaulting to 'pop'")
        test_ids.append(file_id)
        test_preds.append("pop")

# 4. Create Submission DataFrame
submission = pd.DataFrame({
    'id': test_ids,
    'genre': test_preds
})

# 5. Validation Checks (Ensures Kaggle won't reject the file format)
assert len(submission) == 3020, f"Expected 3020 predictions, got {len(submission)}"
assert 'id' in submission.columns and 'genre' in submission.columns, "Missing required columns"

valid_genres = set(all_genres)
if not set(submission['genre']).issubset(valid_genres):
    print("⚠️ Warning: Unknown genres found. Fixing formatting...")
    submission['genre'] = submission['genre'].str.lower().str.strip()

# 6. Save to Disk
submission.to_csv(SUBMISSION_FILE, index=False)
print(f"\n✅ SUCCESS: Submission saved to {SUBMISSION_FILE}")
print(submission.head())

# Optional: W&B Cleanup
try:
    wandb.log({"submission_generated": True})
    wandb.finish()
except:
    pass